# Opening files and Data Cleaning

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon
from geoalchemy2 import Geometry, WKTElement
import matplotlib.pyplot as plt

## Businesses 
Number of businesses by industry and SA2 region, reported by turnover size ranges

In [23]:
businesses = pd.read_csv("Businesses.csv", delimiter=",", engine='python')
print(businesses.shape)
businesses.head()

(12217, 11)


,industry_code,industry_name,sa2_code,sa2_name,0_to_50k_businesses,50k_to_200k_businesses,200k_to_2m_businesses,2m_to_5m_businesses,5m_to_10m_businesses,10m_or_more_businesses,total_businesses
0,A,"Agriculture, Forestry and Fishing",101021007,Braidwood,136,92,63,4,0,0,296
1,A,"Agriculture, Forestry and Fishing",101021008,Karabar,6,3,0,0,0,0,9
2,A,"Agriculture, Forestry and Fishing",101021009,Queanbeyan,6,4,3,0,0,3,15
3,A,"Agriculture, Forestry and Fishing",101021010,Queanbeyan - East,0,3,0,0,0,0,3
4,A,"Agriculture, Forestry and Fishing",101021012,Queanbeyan West - Jerrabomberra,7,4,5,0,0,0,16


In [4]:
businesses.describe()

,sa2_code,0_to_50k_businesses,50k_to_200k_businesses,200k_to_2m_businesses,2m_to_5m_businesses,5m_to_10m_businesses,10m_or_more_businesses,total_businesses
count,1.221700e+04,12217.000000,12217.000000,12217.000000,12217.000000,12217.000000,12217.000000,12217.000000
mean,1.149587e+08,18.822870,22.797659,23.555947,2.980110,1.089711,1.282639,70.540313
std,8.810935e+06,51.385349,43.099939,60.411508,14.196956,6.613522,15.953875,175.595935
min,1.010210e+08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.080112e+08,3.000000,3.000000,3.000000,0.000000,0.000000,0.000000,8.000000
50%,1.160113e+08,8.000000,10.000000,10.000000,0.000000,0.000000,0.000000,33.000000
75%,1.220214e+08,20.000000,26.000000,26.000000,3.000000,0.000000,0.000000,80.000000
max,1.999995e+08,3589.000000,1680.000000,3782.000000,811.000000,458.000000,1504.000000,10125.000000


In [5]:
businesses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12217 entries, 0 to 12216
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   industry_code           12217 non-null  object
 1   industry_name           12217 non-null  object
 2   sa2_code                12217 non-null  int64 
 3   sa2_name                12217 non-null  object
 4   0_to_50k_businesses     12217 non-null  int64 
 5   50k_to_200k_businesses  12217 non-null  int64 
 6   200k_to_2m_businesses   12217 non-null  int64 
 7   2m_to_5m_businesses     12217 non-null  int64 
 8   5m_to_10m_businesses    12217 non-null  int64 
 9   10m_or_more_businesses  12217 non-null  int64 
 10  total_businesses        12217 non-null  int64 
dtypes: int64(8), object(3)
memory usage: 1.0+ MB


### Null values

In [10]:
print("Number of null values:")
for column, null_count in businesses.isnull().sum().items():
    if null_count != 0:
        print(f"{column}:   {null_count},    {int(null_count/len(businesses)*100)}%")
if businesses.isnull().sum().sum() == 0:
    print("Nice! There are no missing values.")


Number of null values:
Nice! There are no missing values.


### No duplicate rows

In [11]:
duplicate_rows = businesses.duplicated()
if duplicate_rows.any():
    print("There are duplicate rows in the DataFrame.")
else:
    print("There are no duplicate rows in the DataFrame.")

There are no duplicate rows in the DataFrame.


### No duplicate sa2 code

In [14]:
column_name = 'sa2_code'

duplicate_values_in_column = businesses.duplicated(subset=[column_name])

if duplicate_values_in_column.any():
    print(f"There are duplicate values in the column '{column_name}'.")
else:
    print(f"There are no duplicate values in the column '{column_name}'.")


There are duplicate values in the column 'sa2_code'.


### Duplicate sa2 names

In [17]:
column_name = 'sa2_name'

duplicate_values_in_column = businesses.duplicated(subset=[column_name])

if duplicate_values_in_column.any():
    print(f"There are duplicate values in the column '{column_name}'.")
else:
    print(f"There are no duplicate values in the column '{column_name}'.")

There are duplicate values in the column 'sa2_name'.


### Industry codes 

In [18]:
unique_classes_count = businesses['industry_code'].nunique()
class_counts = businesses['industry_code'].value_counts()

print("Number of unique classes:", unique_classes_count)
print("\nCounts of each class:")
print(class_counts)

Number of unique classes: 19

Counts of each class:
A    643
K    643
R    643
Q    643
P    643
O    643
N    643
M    643
L    643
J    643
B    643
I    643
H    643
G    643
F    643
E    643
D    643
C    643
S    643
Name: industry_code, dtype: int64


### Industry names

In [22]:
unique_classes_count = businesses['industry_name'].nunique()
class_counts = businesses['industry_name'].value_counts()

print("Number of unique classes:", unique_classes_count)
print("\nCounts of each class:")
print(class_counts)

Number of unique classes: 19

Counts of each class:
Agriculture, Forestry and Fishing                  643
Financial and Insurance Services                   643
Arts and Recreation Services                       643
Health Care and Social Assistance                  643
Education and Training                             643
Public Administration and Safety                   643
Administrative and Support Services                643
Professional, Scientific and Technical Services    643
Rental, Hiring and Real Estate Services            643
Information Media and Telecommunications           643
Mining                                             643
Transport, Postal and Warehousing                  643
Accommodation and Food Services                    643
Retail Trade                                       643
Wholesale Trade                                    643
Construction                                       643
Electricity, Gas, Water and Waste Services         643
Manufacturing

### Count of businesses

In [26]:

sums = businesses.iloc[:, 4:].sum()
print("Sum of each column:")
print(sums)


Sum of each column:
0_to_50k_businesses       229959
50k_to_200k_businesses    278519
200k_to_2m_businesses     287783
2m_to_5m_businesses        36408
5m_to_10m_businesses       13313
10m_or_more_businesses     15670
total_businesses          861791
dtype: int64


#### Observations:



#### Suggestions:

